In [99]:
import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as pltd
from transformers import pipelines, Trainer, TrainingArguments
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset # A Blueprint form Pytoch, explaining standard way of feeding data to a model
import torch # used to convert labels into a format the models understand 
from PIL import Image
import shutil

In [100]:
classes = ["bcc", "mel", "nv", "bkl"]

## Acessing the complete Original Dataset

In [101]:
checkpoint = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(checkpoint)

In [102]:
label2id = {"akiec":0, "bcc":1, "bkl":2, "df":3, "mel":4, "nv":5, "vasc":6}
id2label = {value:key for key,value in label2id.items()}

In [103]:
# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\tle72\.cache\kagglehub\datasets\kmader\skin-cancer-mnist-ham10000\versions\2


#### There are two folders for images, however, only 1 csv file. If we want to remove a class, we have to search within each folder of the kaggle

In [104]:
print(os.listdir(path))

['HAM10000_images_part_1', 'HAM10000_images_part_2', 'HAM10000_metadata.csv', 'hmnist_28_28_L.csv', 'hmnist_28_28_RGB.csv', 'hmnist_8_8_L.csv', 'hmnist_8_8_RGB.csv']


In [105]:
csv_path = os.path.join(path, "HAM10000_metadata.csv")
data = pd.read_csv(csv_path)
data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


## Reducing to 4 classes

In [106]:
sorted_count = data.groupby(["dx"]).size()
sorted_count.head(1000)

dx
akiec     327
bcc       514
bkl      1099
df        115
mel      1113
nv       6705
vasc      142
dtype: int64

In [107]:
# Filtering dataframe to only include the 4 classes. 
# For simplicity we are grouping the Benine Keratosis (bkl) as Kertosis
filtered_data = data[data["dx"].isin(classes)]
filtered_data = filtered_data.reset_index(drop=True)
filtered_data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [108]:
# groups the data based on classes
# Within each class, count the number of lesions
filtered_count = filtered_data.groupby(["dx"])["lesion_id"].count()  ## same --> filtered_data.groupby(["dx"]).size()
filtered_count.head(1000)

dx
bcc     514
bkl    1099
mel    1113
nv     6705
Name: lesion_id, dtype: int64

## Balance the Dataset

In [109]:
image_dir1 = os.path.join(path, "HAM10000_images_part_1")
image_dir2 = os.path.join(path, "HAM10000_images_part_2")

def get_image_path(image_id):
    for i in [image_dir1, image_dir2]:   # loop through 2 folders at once
        p = os.path.join(i, image_id + ".jpg")
        if os.path.exists(p):
            return p
    return None

In [110]:
smallest_count = filtered_data.groupby(["dx"])["lesion_id"].count().min()  

In [111]:
for i in classes:
    if filtered_data[ filtered_data["dx"] == i ]["lesion_id"].count()  > smallest_count:
        current_count = filtered_data[ filtered_data["dx"] == i ]["lesion_id"].count() 
        removable_index = filtered_data[ filtered_data["dx"] == i ]["lesion_id"].sample(n= current_count - smallest_count, random_state=42).index.to_numpy()
        filtered_data = filtered_data.drop(removable_index) 
filtered_data = filtered_data.reset_index(drop=True)
filtered_data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
1,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
2,HAM_0005132,ISIC_0025837,bkl,histo,70.0,female,back
3,HAM_0004234,ISIC_0029396,bkl,histo,85.0,female,chest
4,HAM_0001949,ISIC_0025767,bkl,histo,70.0,male,trunk


In [112]:
# groups the data based on classes
# Within each class, count the number of lesions
filtered_count = filtered_data.groupby(["dx"])["lesion_id"].count()  ## same --> filtered_data.groupby(["dx"]).size()
filtered_count.head(1000)

dx
bcc    514
bkl    514
mel    514
nv     514
Name: lesion_id, dtype: int64

## Copy the balance data set into a local folder

In [123]:
image_series = filtered_data["image_id"]

In [134]:
c_directory = os.getcwd()
d_directory = os.path.join(c_directory, "data","HAM10000")
print(d_directory)

C:\Users\tle72\Downloads\scratch paper\Visual_Transformers\data\HAM10000


In [136]:
for image_id in image_series:
    src_path = get_image_path(image_id) 
    dest_path = os.path.join(d_directory, image_id + ".jpg")
    shutil.copy(src_path, dest_path)

In [135]:
# Test Run -- Do not Run
image_id = image_series[0]
src_path = get_image_path(image_id) 
dest_path = os.path.join(d_directory, image_id + ".jpg")
print(src_path)
print(dest_path)


C:\Users\tle72\.cache\kagglehub\datasets\kmader\skin-cancer-mnist-ham10000\versions\2\HAM10000_images_part_1\ISIC_0025030.jpg
C:\Users\tle72\Downloads\scratch paper\Visual_Transformers\data\HAM10000\ISIC_0025030.jpg


'C:\\Users\\tle72\\.cache\\kagglehub\\datasets\\kmader\\skin-cancer-mnist-ham10000\\versions\\2\\HAM10000_images_part_1\\ISIC_0025030.jpg'